In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix= stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [7]:
X, Y

(tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         [ 5, 13, 13],
         [13, 13,  1],
         [ 0,  0,  0],
         [ 0,  0, 15],
         [ 0, 15, 12],
         [15, 12,  9],
         [12,  9, 22],
         [ 9, 22,  9],
         [22,  9,  1],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1, 22],
         [ 1, 22,  1],
         [ 0,  0,  0],
         [ 0,  0,  9],
         [ 0,  9, 19],
         [ 9, 19,  1],
         [19,  1,  2],
         [ 1,  2,  5],
         [ 2,  5, 12],
         [ 5, 12, 12],
         [12, 12,  1],
         [ 0,  0,  0],
         [ 0,  0, 19],
         [ 0, 19, 15],
         [19, 15, 16],
         [15, 16,  8],
         [16,  8,  9],
         [ 8,  9,  1]]),
 tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
          1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0]))

In [8]:
# embedding lookup table C:
# in the paper: 17K words in 30 dim spqce, Here as we have only 27, ok to start with a 2 dim embedding


In [9]:
C = torch.randn((27, 2))

In [10]:
C

tensor([[-0.1191, -0.0405],
        [ 1.1354, -0.5030],
        [-0.1611,  0.3256],
        [ 1.2891, -0.6481],
        [-0.2725,  0.6902],
        [ 0.6419,  0.9264],
        [-0.3312,  0.0764],
        [ 0.2555,  1.2842],
        [ 0.9330, -0.2151],
        [ 0.8814,  0.9063],
        [ 0.5054, -0.3409],
        [-1.7259, -0.2348],
        [ 0.2924, -1.2340],
        [ 0.4266,  0.3458],
        [ 1.5144, -0.3638],
        [ 0.9527,  0.8028],
        [-1.1959, -1.0080],
        [-0.0416,  0.7825],
        [ 0.3616,  0.4097],
        [ 0.0177,  0.3731],
        [-1.6607, -0.3190],
        [ 0.0049,  0.8762],
        [-0.8987,  0.3931],
        [-0.9269,  0.1591],
        [-1.8954, -0.9242],
        [-0.6452, -1.4625],
        [ 0.1749,  0.6618]])

In [11]:
C.shape

torch.Size([27, 2])

In [12]:
# before embedding all integers inside the input X, we make an example with 5
# one way is to index 5 in the lookup table C, getting the 5th row
C[5]

tensor([0.6419, 0.9264])

In [13]:
# the other way is to use one-hot encoding. Output is identical. All zero masking out all the rows but the 5th
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([0.6419, 0.9264])

In [14]:
 # using indexing to do embedding is equal to use one-hot encoding. Here we use index as it is much faster.
# this can be seen as first layer of NN.

In [15]:
emb = C[X]

In [16]:
emb.shape

torch.Size([32, 3, 2])

In [17]:
X, C, emb

(tensor([[ 0,  0,  0],
         [ 0,  0,  5],
         [ 0,  5, 13],
         [ 5, 13, 13],
         [13, 13,  1],
         [ 0,  0,  0],
         [ 0,  0, 15],
         [ 0, 15, 12],
         [15, 12,  9],
         [12,  9, 22],
         [ 9, 22,  9],
         [22,  9,  1],
         [ 0,  0,  0],
         [ 0,  0,  1],
         [ 0,  1, 22],
         [ 1, 22,  1],
         [ 0,  0,  0],
         [ 0,  0,  9],
         [ 0,  9, 19],
         [ 9, 19,  1],
         [19,  1,  2],
         [ 1,  2,  5],
         [ 2,  5, 12],
         [ 5, 12, 12],
         [12, 12,  1],
         [ 0,  0,  0],
         [ 0,  0, 19],
         [ 0, 19, 15],
         [19, 15, 16],
         [15, 16,  8],
         [16,  8,  9],
         [ 8,  9,  1]]),
 tensor([[-0.1191, -0.0405],
         [ 1.1354, -0.5030],
         [-0.1611,  0.3256],
         [ 1.2891, -0.6481],
         [-0.2725,  0.6902],
         [ 0.6419,  0.9264],
         [-0.3312,  0.0764],
         [ 0.2555,  1.2842],
         [ 0.9330, -0.2151],
 

In [18]:
emb

tensor([[[-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405]],

        [[-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [ 0.6419,  0.9264]],

        [[-0.1191, -0.0405],
         [ 0.6419,  0.9264],
         [ 0.4266,  0.3458]],

        [[ 0.6419,  0.9264],
         [ 0.4266,  0.3458],
         [ 0.4266,  0.3458]],

        [[ 0.4266,  0.3458],
         [ 0.4266,  0.3458],
         [ 1.1354, -0.5030]],

        [[-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405]],

        [[-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [ 0.9527,  0.8028]],

        [[-0.1191, -0.0405],
         [ 0.9527,  0.8028],
         [ 0.2924, -1.2340]],

        [[ 0.9527,  0.8028],
         [ 0.2924, -1.2340],
         [ 0.8814,  0.9063]],

        [[ 0.2924, -1.2340],
         [ 0.8814,  0.9063],
         [-0.8987,  0.3931]],

        [[ 0.8814,  0.9063],
         [-0.8987,  0.3931],
         [ 0.8814,  0.9063]],

        [[-0.8987,  0

In [19]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)

tensor([[-0.1191, -0.0405, -0.1191, -0.0405, -0.1191, -0.0405],
        [-0.1191, -0.0405, -0.1191, -0.0405,  0.6419,  0.9264],
        [-0.1191, -0.0405,  0.6419,  0.9264,  0.4266,  0.3458],
        [ 0.6419,  0.9264,  0.4266,  0.3458,  0.4266,  0.3458],
        [ 0.4266,  0.3458,  0.4266,  0.3458,  1.1354, -0.5030],
        [-0.1191, -0.0405, -0.1191, -0.0405, -0.1191, -0.0405],
        [-0.1191, -0.0405, -0.1191, -0.0405,  0.9527,  0.8028],
        [-0.1191, -0.0405,  0.9527,  0.8028,  0.2924, -1.2340],
        [ 0.9527,  0.8028,  0.2924, -1.2340,  0.8814,  0.9063],
        [ 0.2924, -1.2340,  0.8814,  0.9063, -0.8987,  0.3931],
        [ 0.8814,  0.9063, -0.8987,  0.3931,  0.8814,  0.9063],
        [-0.8987,  0.3931,  0.8814,  0.9063,  1.1354, -0.5030],
        [-0.1191, -0.0405, -0.1191, -0.0405, -0.1191, -0.0405],
        [-0.1191, -0.0405, -0.1191, -0.0405,  1.1354, -0.5030],
        [-0.1191, -0.0405,  1.1354, -0.5030, -0.8987,  0.3931],
        [ 1.1354, -0.5030, -0.8987,  0.3

In [20]:
W1 = torch.randn((6, 100)) 
b1 = torch.randn(100)
# hidden layer construction - initialized with random parameters
# 6 is the number of inputs given from 3 vectors projected on 2D spaced (2-dim embedding)
# 100 is the number of neurons and we can chose by us 

In [21]:
# problem: emb @ W1 + b1 cannot be done as emb is torch.Size([32, 3, 2]) and W1 is (6, 100)
# in emb, 3 vector concatenated (torch.cat)so that emb becomes (32, 6)
# the definition above is not generic and need to be rewritten if block size changes
# a generic espression is the following:
# len(torch.unbind(emb,1)) is 3 -> returning 3 tensors: one for index 0, one for index 1 and one for index 2
#torch.cat(torch.unbind(emb,1), 1).shape
torch.unbind(emb,1)



(tensor([[-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [ 0.6419,  0.9264],
         [ 0.4266,  0.3458],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [ 0.9527,  0.8028],
         [ 0.2924, -1.2340],
         [ 0.8814,  0.9063],
         [-0.8987,  0.3931],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [ 1.1354, -0.5030],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [ 0.8814,  0.9063],
         [ 0.0177,  0.3731],
         [ 1.1354, -0.5030],
         [-0.1611,  0.3256],
         [ 0.6419,  0.9264],
         [ 0.2924, -1.2340],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [ 0.0177,  0.3731],
         [ 0.9527,  0.8028],
         [-1.1959, -1.0080],
         [ 0.9330, -0.2151]]),
 tensor([[-0.1191, -0.0405],
         [-0.1191, -0.0405],
         [ 0

In [22]:
torch.cat(torch.unbind(emb,1), 1)

tensor([[-0.1191, -0.0405, -0.1191, -0.0405, -0.1191, -0.0405],
        [-0.1191, -0.0405, -0.1191, -0.0405,  0.6419,  0.9264],
        [-0.1191, -0.0405,  0.6419,  0.9264,  0.4266,  0.3458],
        [ 0.6419,  0.9264,  0.4266,  0.3458,  0.4266,  0.3458],
        [ 0.4266,  0.3458,  0.4266,  0.3458,  1.1354, -0.5030],
        [-0.1191, -0.0405, -0.1191, -0.0405, -0.1191, -0.0405],
        [-0.1191, -0.0405, -0.1191, -0.0405,  0.9527,  0.8028],
        [-0.1191, -0.0405,  0.9527,  0.8028,  0.2924, -1.2340],
        [ 0.9527,  0.8028,  0.2924, -1.2340,  0.8814,  0.9063],
        [ 0.2924, -1.2340,  0.8814,  0.9063, -0.8987,  0.3931],
        [ 0.8814,  0.9063, -0.8987,  0.3931,  0.8814,  0.9063],
        [-0.8987,  0.3931,  0.8814,  0.9063,  1.1354, -0.5030],
        [-0.1191, -0.0405, -0.1191, -0.0405, -0.1191, -0.0405],
        [-0.1191, -0.0405, -0.1191, -0.0405,  1.1354, -0.5030],
        [-0.1191, -0.0405,  1.1354, -0.5030, -0.8987,  0.3931],
        [ 1.1354, -0.5030, -0.8987,  0.3

In [23]:
# more efficient way to do it by using a view: there is no memory change of the tensor
# the a.storage() stays the same. Changing only attributes to represent the tensor

In [24]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [25]:
a.shape

torch.Size([18])

In [26]:
a.view(2,9)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [27]:
a.storage()

/var/folders/6x/g47f8b917pv3w7c2_dccwxw40000gn/T/ipykernel_49668/214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [28]:
a.untyped_storage()

 0
 0
 0
 0
 0
 0
 0
 0
 1
 0
 0
 0
 0
 0
 0
 0
 2
 0
 0
 0
 0
 0
 0
 0
 3
 0
 0
 0
 0
 0
 0
 0
 4
 0
 0
 0
 0
 0
 0
 0
 5
 0
 0
 0
 0
 0
 0
 0
 6
 0
 0
 0
 0
 0
 0
 0
 7
 0
 0
 0
 0
 0
 0
 0
 8
 0
 0
 0
 0
 0
 0
 0
 9
 0
 0
 0
 0
 0
 0
 0
 10
 0
 0
 0
 0
 0
 0
 0
 11
 0
 0
 0
 0
 0
 0
 0
 12
 0
 0
 0
 0
 0
 0
 0
 13
 0
 0
 0
 0
 0
 0
 0
 14
 0
 0
 0
 0
 0
 0
 0
 15
 0
 0
 0
 0
 0
 0
 0
 16
 0
 0
 0
 0
 0
 0
 0
 17
 0
 0
 0
 0
 0
 0
 0
[torch.storage.UntypedStorage(device=cpu) of size 144]

In [29]:
emb.shape

torch.Size([32, 3, 2])

In [30]:
emb.view(32,6)

tensor([[-0.1191, -0.0405, -0.1191, -0.0405, -0.1191, -0.0405],
        [-0.1191, -0.0405, -0.1191, -0.0405,  0.6419,  0.9264],
        [-0.1191, -0.0405,  0.6419,  0.9264,  0.4266,  0.3458],
        [ 0.6419,  0.9264,  0.4266,  0.3458,  0.4266,  0.3458],
        [ 0.4266,  0.3458,  0.4266,  0.3458,  1.1354, -0.5030],
        [-0.1191, -0.0405, -0.1191, -0.0405, -0.1191, -0.0405],
        [-0.1191, -0.0405, -0.1191, -0.0405,  0.9527,  0.8028],
        [-0.1191, -0.0405,  0.9527,  0.8028,  0.2924, -1.2340],
        [ 0.9527,  0.8028,  0.2924, -1.2340,  0.8814,  0.9063],
        [ 0.2924, -1.2340,  0.8814,  0.9063, -0.8987,  0.3931],
        [ 0.8814,  0.9063, -0.8987,  0.3931,  0.8814,  0.9063],
        [-0.8987,  0.3931,  0.8814,  0.9063,  1.1354, -0.5030],
        [-0.1191, -0.0405, -0.1191, -0.0405, -0.1191, -0.0405],
        [-0.1191, -0.0405, -0.1191, -0.0405,  1.1354, -0.5030],
        [-0.1191, -0.0405,  1.1354, -0.5030, -0.8987,  0.3931],
        [ 1.1354, -0.5030, -0.8987,  0.3

In [31]:
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
h

# emb.view(32,6) can be replaced by emb.view(-1,6) -> torch understand the size automatically

tensor([[-0.2501,  0.7405, -0.5952,  ...,  0.7457,  0.6259, -0.4331],
        [ 0.9921,  0.9650,  0.0375,  ...,  0.9236,  0.9333, -0.6806],
        [ 0.9563,  0.9996,  0.1882,  ...,  0.9626,  0.9969, -0.0012],
        ...,
        [ 0.5434, -0.7805, -0.6074,  ..., -0.9948,  0.2709, -0.9937],
        [ 0.9998,  0.9919,  0.1386,  ...,  0.9975,  0.9920,  0.0054],
        [ 0.8929,  0.9999,  0.7250,  ..., -0.5728,  0.9998, -0.9873]])

In [32]:
h.shape

torch.Size([32, 100])

In [33]:
# broadcasting works here too
# emb @ W1 is (32, 100) and b1 is (100) -> align to the right, fake dimension 1, same b1 added to all the rows
# 32, 100
#  1 , 100

In [34]:
# create 2nd and final hidden layer - input: 100 and output 27: possible characters
W2 = torch.randn((100, 27)) 
b2 = torch.randn(27)

In [35]:
logits = h @ W2 + b2

In [36]:
logits.shape

torch.Size([32, 27])

In [37]:
logits

tensor([[-8.7769e+00, -3.4026e-01,  1.2337e+00, -8.3989e-01,  7.1166e+00,
          7.6485e+00,  1.2448e+00,  2.9880e+00,  3.7268e+00, -7.9384e+00,
         -1.0980e+01,  6.8228e+00,  4.3521e+00, -6.8728e+00,  4.7269e+00,
          5.1717e+00, -9.7000e-01, -4.2502e+00, -3.8576e+00, -5.3782e+00,
          4.7777e+00, -1.1259e+00, -1.5503e+00, -4.1298e+00,  7.6459e+00,
         -1.0096e+01,  5.8924e+00],
        [-1.6632e+01,  2.5229e+00,  8.2488e+00, -9.0375e+00,  1.5999e+01,
          1.5425e+00, -5.0745e-01,  2.7695e+00,  5.9206e+00, -2.1320e+01,
         -1.5764e+01,  3.9077e+00,  1.0437e+01, -2.4794e+00, -1.3543e+01,
          7.2046e+00,  2.8943e+00,  5.4746e+00, -3.2026e+00, -6.2551e+00,
          5.2662e+00, -6.0230e+00, -3.3741e+00,  4.2660e+00,  5.6389e+00,
         -1.4770e+01,  6.0555e+00],
        [-5.9024e+00,  6.0555e+00, -2.4931e+00, -5.4489e+00,  9.4657e+00,
          2.8608e+00, -7.6058e+00,  4.4435e+00, -3.4007e+00, -1.9728e+01,
         -1.6848e+01,  4.1479e+00,  8.21

In [38]:
counts = logits.exp()

In [39]:
prob = counts / counts.sum(1, keepdims=True)

In [40]:
prob.shape

torch.Size([32, 27])

In [41]:
prob[0].sum()

tensor(1.0000)

In [42]:
loss = -prob[torch.arange(32), Y].log().mean()
loss
# for many chars, it does not work well as the network thinks they're very unlikely
# this because the network hasn't been trained yet 
# we want to maximize the prob -> minimize the negative log_loss

tensor(16.6326)

In [43]:
# index in the rows of prob and each row to plug out the prob of the correct char given by Y
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [44]:
# Y is the array created showing the next char we'd like to predict
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [45]:
# --------- made respectable -------------

In [46]:
X.shape, Y.shape # dataset

(torch.Size([32, 3]), torch.Size([32]))

In [47]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g, requires_grad=True)
W1 = torch.randn((6, 100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn((100, 27), generator=g, requires_grad=True)
b2 = torch.randn(27, generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]

In [48]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [49]:
# emb = C[X] # (32, 3, 2)
# h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
# logits = h @ W2 + b2 # (32, 27)
# counts = logits.exp()
# prob = counts / counts.sum(1, keepdims=True)
# loss = -prob[torch.arange(32), Y].log().mean()
# loss

In [50]:
F.cross_entropy(logits, Y)
# advantages: forward and backward passes more efficient + behaves better numerically (esp. when extremely high logits)

tensor(16.6326)

In [51]:
for p in parameters:
    p.requires_grad = True

In [52]:
for p in parameters:
    print(p.requires_grad) 

True
True
True
True
True


In [59]:
for _ in range(1000):
    # forward pass 
    emb = C[X] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y)
    # print(loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    for p in parameters:
        p.data += -0.1 * p.grad
print(loss.item())

0.2546193301677704


In [64]:
logits.max(1)

torch.return_types.max(
values=tensor([13.6006, 18.3582, 20.8519, 20.9889, 17.1317, 13.6006, 16.3810, 14.4958,
        16.2390, 18.8178, 16.3761, 21.3384, 13.6006, 17.5940, 17.5711, 20.5228,
        13.6006, 16.9799, 15.6157, 17.5291, 18.9370, 16.4204, 11.2760, 11.0386,
        15.7806, 13.6006, 16.5394, 17.3466, 13.0147, 16.5028, 19.5491, 16.5745],
       grad_fn=<MaxBackward0>),
indices=tensor([19, 13, 13,  1,  0, 19, 12,  9, 22,  9,  1,  0, 19, 22,  1,  0, 19, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0]))

In [66]:
Y
# overfitting: 3481 parameters for 32 inputs; where 1:1 inputs:outputs, prediction is OK
# first one not perfect as ... can come before several characters

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [67]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
    # print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix= stoi[ch]
        X.append(context)
        Y.append(ix)
      #  print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [68]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [69]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g, requires_grad=True)
W1 = torch.randn((6, 100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn((100, 27), generator=g, requires_grad=True)
b2 = torch.randn(27, generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]

In [70]:
for p in parameters:
    p.requires_grad = True

In [71]:
for _ in range(10):
    # forward pass 
    emb = C[X] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y)
    print(loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    # update
    for p in parameters:
        p.data += -0.1 * p.grad
# print(loss.item())

19.505229949951172
17.084510803222656
15.776554107666016
14.833368301391602
14.002623558044434
13.25327205657959
12.579926490783691
11.983114242553711
11.470508575439453
11.051871299743652
